# Backbone Verification Template

Drop in any `make_*` factory and run all evaluators end-to-end to verify a new backbone.

**Steps:**
1. Fill in the config cell — set `MODEL_NAME`, `EMBEDDING_CACHE`, and `HAS_TEXT_ENCODER`.
2. Load your backbone in the "Backbone" cell.
3. Run all cells top-to-bottom.

Variable contract used throughout:
- `transform` — MONAI `Compose` for the dataset
- `encoder` — `ImageEncoderWrapper` (or any `nn.Module` with `forward(imgs) → Tensor[B, D]`)
- `text_encoder`, `tokenizer` — required only when `HAS_TEXT_ENCODER = True`

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_IMAGE_DIR = "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/train/"
TEST_DATASET_IMAGE_DIR = "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
DATASET_NAME      = "vindr-cxr"   # used for cache paths and output dir naming

CACHE_DIR = "/path/to/cache/"

# ── Backbone ──────────────────────────────────────────────────────────────────
MODEL_NAME      = "chexfound"   # used for output paths and MONAI cache dir
EMBEDDING_CACHE = f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}.pkl"  # set to None to disable

OUT_DIR    = f"./eval_out/verify_{MODEL_NAME}_{DATASET_NAME}"
DEVICE     = "cuda:6"
BATCH_SIZE = 32

# ── Evaluation ────────────────────────────────────────────────────────────────
LABEL_SUBSET = ["aortic_enlargement", "atelectasis", "calcification", "cardiomegaly", "clavicle_fracture", "consolidation", "copd", "edema", "emphysema", "enlarged_pa", "ild", "infiltration", "lung_cavity", "lung_cyst", "lung_opacity", "lung_tumor", "mediastinal_shift", "no_finding", "nodule/mass", "other_diseases", "other_lesion", "pleural_effusion", "pleural_thickening", "pneumonia", "pneumothorax", "pulmonary_fibrosis", "rib_fracture", "tuberculosis"]   # None = all labels from the dataset
              # or e.g. ["cardiomegaly", "pleural_effusion"] for a subset

N_FOLDS         = 5
N_TRAIN_SAMPLES = None   # None = full fold train split; [200, 500, 1500] = sweep

LP_C = 0.1

# Set True if your backbone has a text tower — enables the zero-shot section.
HAS_TEXT_ENCODER = False

## 2. Backbone

Replace the example below with your own `make_*` factory call.
All downstream cells use `transform`, `encoder`, `text_encoder`, and `tokenizer`.

In [ ]:
# ── Image-only backbone example (RAD-DINO) ───────────────────────────────
# from radharmony.evaluator.backbones import make_raddino
# transform, encoder = make_raddino(device=DEVICE)
# text_encoder = tokenizer = None

# ── Vision-language backbone example (MedSigLIP) ─────────────────────────
# from radharmony.evaluator.backbones import make_medsiglip
# transform, encoder, text_encoder, processor = make_medsiglip(device=DEVICE)
# tokenizer = lambda texts: processor(text=texts, return_tensors="pt", padding="max_length")
# ─────────────────────────────────────────────────────────────────────────

from radharmony.evaluator.backbones import make_chexfound
transform, encoder = make_chexfound(device=DEVICE)

print(f"Encoder:  {type(encoder).__name__}")
print(f"Backbone: {type(encoder.model).__name__}")

## 3. Dataset

In [ ]:
# ── Fill in your dataset ──────────────────────────────────────────────────────
# Import any RadHarmony dataset class that inherits BaseRadiologicalDataset
# and supports output_cls=True.  Pass the matching constructor args.
#
# The only required wiring here is:
#   transform=transform   ← the MONAI Compose from the backbone cell above
#   output_cls=True       ← exposes the "cls" tensor for evaluators
#
# VinDr-CXR (default):
# from radharmony.dataset import VinDrCXRTrainDataset

# ds = VinDrCXRTrainDataset(
#     base_image_dir=DATASET_IMAGE_DIR,
#     transform=transform,
#     cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
#     output_cls=True,
# )

from radharmony.dataset import VinDrCXRTrainDataset, VinDrCXRTestDataset

ds = VinDrCXRTrainDataset(
    base_image_dir=DATASET_IMAGE_DIR,
    transform=transform,
    cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
    output_cls=True,
)

test_ds = VinDrCXRTestDataset(
    base_image_dir=TEST_DATASET_IMAGE_DIR,
    transform=transform,
    cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
    output_cls=True,
)
    

# ── Other dataset examples (uncomment + fill paths) ───────────────────────────
# from radharmony.dataset import ChestXray14Dataset
# ds = ChestXray14Dataset(
#     base_image_dir=DATASET_IMAGE_DIR,
#     transform=transform,
#     cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
#     output_cls=True,
# )

# from radharmony.dataset import CheXpertTrainDataset
# ds = CheXpertTrainDataset(
#     base_image_dir=DATASET_IMAGE_DIR,
#     transform=transform,
#     cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
#     output_cls=True,
# )

# from radharmony.dataset import PadChestDataset
# ds = PadChestDataset(
#     base_image_dir=DATASET_IMAGE_DIR,
#     transform=transform,
#     cache_dir=f"{CACHE_DIR}{MODEL_NAME}_{DATASET_NAME}/",
#     output_cls=True,
# )

_labels_to_eval = LABEL_SUBSET if LABEL_SUBSET is not None else ds.LABEL_COLS
print(f"Dataset:  {type(ds).__name__}  ({len(ds.get_harmonized_df())} images)")
print(f"Labels ({len(_labels_to_eval)}): {_labels_to_eval}")

In [ ]:
torch_ds = ds.get_datasets()

In [ ]:
df = ds.get_harmonized_df()
df

In [ ]:
df[["study_id"]+LABEL_SUBSET].groupby('study_id').sum().sum()/len(df)

In [ ]:
import matplotlib.pyplot as plt

idx = 78
img = torch_ds[idx]["img"].float()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img.numpy()[0], cmap="gray")
ax.set_title(
    f"{MODEL_NAME}\nmin={img.min():.3f}  max={img.max():.3f}\n"
    f"mean={img.mean():.3f}  std={img.std():.3f}",
    fontsize=9,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. LinearProbeEvaluator

Per-label logistic regression on frozen features. **K-fold mode**: embeddings are extracted once; `KFold(N_FOLDS)` partitions the feature matrix; variance comes from fold axis.

In [ ]:
from radharmony.evaluator import LinearProbeEvaluator

lp = LinearProbeEvaluator(
    image_encoder=encoder,
    train_dataset=ds,
    test_dataset=test_ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    probe_kwargs={"C": LP_C},
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)

df_lp = lp.evaluate()
lp.save_results(df_lp, output_dir=f"{OUT_DIR}/linear")
df_lp.groupby(["n_train", "label"])[["auroc", "auprc", "f1"]].mean()

## 6. KNNProbeEvaluator

Soft-vote k-NN on L2-normalized features (cosine metric). Reuses the cached embeddings — no extra GPU time needed.

In [ ]:
from radharmony.evaluator import KNNProbeEvaluator

knn = KNNProbeEvaluator(
    image_encoder=encoder,
    dataset=ds,
    labels=LABEL_SUBSET,
    k=20,
    metric="cosine",
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)

df_knn = knn.evaluate()
knn.save_results(df_knn, output_dir=f"{OUT_DIR}/knn")
df_knn.groupby(["n_train", "label"])[["auroc", "auprc", "f1"]].mean()

## 7. PrototypeProbeEvaluator

Per-label nearest-centroid classifier on L2-normalised features. Reuses cached embeddings — no extra GPU extraction.

In [ ]:
from radharmony.evaluator import PrototypeProbeEvaluator

proto = PrototypeProbeEvaluator(
    image_encoder=encoder,
    dataset=ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)

df_proto = proto.evaluate()
proto.save_results(df_proto, output_dir=f"{OUT_DIR}/prototype")
df_proto.groupby(["n_train", "label"])[["auroc", "auprc", "f1"]].mean()

## 8. SVMProbeEvaluator

Per-label `LinearSVC` on frozen features. Reuses cached embeddings — no extra GPU extraction.

In [ ]:
from radharmony.evaluator import SVMProbeEvaluator

svm = SVMProbeEvaluator(
    image_encoder=encoder,
    dataset=ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)

df_svm = svm.evaluate()
svm.save_results(df_svm, output_dir=f"{OUT_DIR}/svm")
df_svm.groupby(["n_train", "label"])[["auroc", "auprc", "f1"]].mean()

## 9. ZeroShotEvaluator

Skipped when `HAS_TEXT_ENCODER = False`. Requires `text_encoder` and `tokenizer` to be defined in the backbone cell.

In [ ]:
# ── Zero-shot prompts ─────────────────────────────────────────────────────────
# Auto-generated from the dataset's label columns.
# Override individual entries below for domain-specific phrasing.

def _label_to_phrase(label: str) -> str:
    return label.lower().replace("_", " ")

_prompt_labels = LABEL_SUBSET if LABEL_SUBSET is not None else ds.LABEL_COLS

PROMPTS = {
    lbl: [
        f"chest X-ray showing {_label_to_phrase(lbl)}",
        f"radiograph with {_label_to_phrase(lbl)}",
    ]
    for lbl in _prompt_labels
}

NEGATIVE_PROMPTS = {
    lbl: [
        f"chest X-ray without {_label_to_phrase(lbl)}",
        f"radiograph showing no {_label_to_phrase(lbl)}",
    ]
    for lbl in _prompt_labels
}

# Override with more precise phrasing for specific labels, e.g.:
# PROMPTS["lung_opacity"]      = ["chest radiograph with pulmonary opacity", "airspace opacity on chest X-ray"]
# PROMPTS["pleural_effusion"]  = ["chest X-ray with pleural effusion", "blunted costophrenic angle on radiograph"]

print(f"Prompts defined for {len(PROMPTS)} labels")

In [ ]:
if not HAS_TEXT_ENCODER:
    print("Skipping zero-shot — set HAS_TEXT_ENCODER=True and define text_encoder / tokenizer.")
else:
    from radharmony.evaluator import ZeroShotEvaluator

    zs = ZeroShotEvaluator(
        image_encoder=encoder,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        prompts=PROMPTS,
        negative_prompts=NEGATIVE_PROMPTS,
        dataset=ds,
        labels=LABEL_SUBSET,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        embedding_cache=EMBEDDING_CACHE,
    )

    df_zs = zs.evaluate()
    zs.save_results(df_zs, output_dir=f"{OUT_DIR}/zeroshot")
    display(df_zs[["label", "auroc", "auprc", "f1"]])

## 10. FinetuneEvaluator (RAD-DINO, head-only)

`freeze_backbone=True` trains only the linear head over frozen features. **K-fold mode**: same fold splits as the probes.

In [ ]:
from radharmony.evaluator import FinetuneEvaluator

ft = FinetuneEvaluator(
    image_encoder=encoder,
    dataset=ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    epochs=20,
    lr=1e-3,
    freeze_backbone=True,
    early_stop_patience=5,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

df_ft = ft.evaluate()
ft.save_results(df_ft, output_dir=f"{OUT_DIR}/finetune")
df_ft.groupby(["n_train", "label"])[["auroc", "auprc", "f1"]].mean()

## 11. Results Summary

In [ ]:
import os
import pandas as pd

runs = [
    ("LinearProbe",    f"{OUT_DIR}/linear/results_summary.csv"),
    ("KNNProbe",       f"{OUT_DIR}/knn/results_summary.csv"),
    ("PrototypeProbe", f"{OUT_DIR}/prototype/results_summary.csv"),
    ("SVMProbe",       f"{OUT_DIR}/svm/results_summary.csv"),
    ("Finetune",       f"{OUT_DIR}/finetune/results_summary.csv"),
]
if HAS_TEXT_ENCODER:
    runs.append(("ZeroShot", f"{OUT_DIR}/zeroshot/results_summary.csv"))

frames = []
for evaluator, path in runs:
    if not os.path.exists(path):
        print(f"  skipping {evaluator} — not found")
        continue
    df = pd.read_csv(path)
    df["evaluator"] = evaluator
    frames.append(df)

summary = pd.concat(frames, ignore_index=True)
macro = summary[summary.label == "macro_average"].copy()
macro["auroc"] = macro["auroc_mean"].round(3).astype(str) + " ± " + macro["auroc_std"].round(3).astype(str)
macro["auprc"] = macro["auprc_mean"].round(3).astype(str) + " ± " + macro["auprc_std"].round(3).astype(str)
macro[["evaluator", "auroc", "auprc"]]